In [1]:
import requests, time, pathlib

In [ ]:
out = pathlib.Path("Structures")
out.mkdir(exist_ok=True)

def parse_pisces(path):
    ids = []
    with open(path) as f:
        next(f)
        for line in f:
            tok = line.split()[0]
            ids.append((tok[:4].lower(), tok[4:])) # (pdb_id, chain)
    return ids

entries = parse_pisces("selected_chains.txt")

for pdb_id in sorted({p for p, i in entries}):
    dest = out / f"{pdb_id}.cif"
    if dest.exists():
        continue
    r = requests.get(f"https://files.rcsb.org/download/{pdb_id}.cif", timeout=30)
    if r.ok:
        dest.write_bytes(r.content)
    time.sleep(0.1)

In [ ]:
from collections import defaultdict
from Bio.PDB import MMCIFParser, DSSP
import pandas as pd
import os

os.environ["LIBCIFPP_DATA_DIR"] = os.path.expanduser("~/.local/dssp/share/libcifpp")
MKDSSP = os.path.expanduser("~/.local/dssp/bin/mkdssp")

three_state = {
    **dict.fromkeys("HGI", "H"),
    **dict.fromkeys("EB", "E"),
}  # everything else -> "C"

# group wanted chains by entry so DSSP runs once per structure
wanted = defaultdict(set)
for pdb_id, chain_id in entries:
    wanted[pdb_id].add(chain_id)

rows = []
for pdb_id, chains in wanted.items():
    path = f"Structures/{pdb_id}.cif"
    try:
        s = MMCIFParser(QUIET=True).get_structure(pdb_id, path)
        dssp = DSSP(s[0], path, dssp=MKDSSP, file_type="MMCIF")
    except Exception as e:
        print(f"skipping {pdb_id}: {e}")
        continue

    seq, ss = defaultdict(list), defaultdict(list)
    for (ch, _), v in dssp.property_dict.items():
        if ch in chains:
            seq[ch].append(v[1])
            ss[ch].append(three_state.get(v[2], "C"))

    for ch in chains:
        if seq[ch]:
            rows.append({
                "pdb_id": pdb_id,
                "chain": ch,
                "sequence": "".join(seq[ch]),
                "structure": "".join(ss[ch]),
            })

df = pd.DataFrame(rows)
df.to_csv("selected_proteins.csv", index=False)

/Users/StevenEngelbretson/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/CS327/project/.venv/lib/python3.13/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is lower than requested, this may cause validation errors
Loaded dictionary does not match name=mmcif_pdbx.dic and version=5.416
Unknown compound: ALA
Unknown compound: ARG
Unknown compound: ASN
Unknown compound: ASP
Unknown compound: CYS
Unknown compound: GLN
Unknown compound: GLU
Unknown compound: GLY
Unknown compound: GOL
Unknown compound: HIS
Unknown compound: HOH
Unknown compound: ILE
Unknown compound: LEU
Unknown compound: LYS
Unknown compound: MET
Unknown compound: PHE
Unknown compound: PRO
Unknown compound: SER
Unknown compound: SF4
Unknown compound: SO4
Unknown compound: THR
Unknown compound: TRP
Unknown compound: TYR
Unknown compound: VAL

  warnings.warn(err)
/Users/StevenEngelbretson/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/CS327/project/.venv/lib/pyt

skipping 7af2: DSSP failed to produce an output


/Users/StevenEngelbretson/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/CS327/project/.venv/lib/python3.13/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Unknown compound: ALA
Unknown compound: ARG
Unknown compound: ASN
Unknown compound: ASP
Unknown compound: CYS
Unknown compound: GLN
Unknown compound: GLU
Unknown compound: GLY
Unknown compound: HIS
Unknown compound: HOH
Unknown compound: ILE
Unknown compound: LEU
Unknown compound: LYS
Unknown compound: MET
Unknown compound: NA
Unknown compound: PHE
Unknown compound: SER
Unknown compound: TYR
Unknown compound: VAL

  warnings.warn(err)
/Users/StevenEngelbretson/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/CS327/project/.venv/lib/python3.13/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Unknown compound: ACT
Unknown compound: ALA
Unknown compound: ARG
Unknown compound: ASN
Unknown compound: ASP
Unknown compound: CYS
Unknown compound: GLN
Unknown compound: GLU
Unknown compound: GLY
Unknown compound: HIS
Unkno

skipping 1p9g: DSSP failed to produce an output


/Users/StevenEngelbretson/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/CS327/project/.venv/lib/python3.13/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Unknown compound: ALA
Unknown compound: ARG
Unknown compound: ASN
Unknown compound: ASP
Unknown compound: BEZ
Unknown compound: GLN
Unknown compound: GLU
Unknown compound: GLY
Unknown compound: GOL
Unknown compound: HIS
Unknown compound: HOH
Unknown compound: ILE
Unknown compound: LEU
Unknown compound: LYS
Unknown compound: MES
Unknown compound: MET
Unknown compound: NA
Unknown compound: PHE
Unknown compound: PRO
Unknown compound: SER
Unknown compound: SO4
Unknown compound: THR
Unknown compound: TRP
Unknown compound: TYR
Unknown compound: VAL

  warnings.warn(err)
/Users/StevenEngelbretson/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/CS327/project/.venv/lib/python3.13/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Unknown compound: ACT
Unknown compound: ALA
Unknown compound: ARG
Unknown compound: ASN
Unkno

skipping 6kfn: DSSP failed to produce an output


... (213 outputs left out)

skipping 4gip: DSSP failed to produce an output


/Users/StevenEngelbretson/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/CS327/project/.venv/lib/python3.13/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Unknown compound: ALA
Unknown compound: ARG
Unknown compound: ASN
Unknown compound: ASP
Unknown compound: CYS
Unknown compound: GLN
Unknown compound: GLU
Unknown compound: GLY
Unknown compound: HIS
Unknown compound: HOH
Unknown compound: ILE
Unknown compound: LEU
Unknown compound: LYS
Unknown compound: MET
Unknown compound: NA
Unknown compound: PHE
Unknown compound: PRO
Unknown compound: SER
Unknown compound: THR
Unknown compound: TRP
Unknown compound: TYR
Unknown compound: VAL
Unknown compound: ZN

  warnings.warn(err)
/Users/StevenEngelbretson/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/CS327/project/.venv/lib/python3.13/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Unknown compound: ALA
Unknown compound: ARG
Unknown compound: ASN
Unknown compound: ASP
Unknown compound: CYS
Unknown compound: DA
Unknown

8586 chains written


/Users/StevenEngelbretson/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/CS327/project/.venv/lib/python3.13/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Unknown compound: ALA
Unknown compound: ARG
Unknown compound: ASN
Unknown compound: ASP
Unknown compound: CYS
Unknown compound: GLN
Unknown compound: GLU
Unknown compound: GLY
Unknown compound: GNP
Unknown compound: HIS
Unknown compound: HOH
Unknown compound: ILE
Unknown compound: LEU
Unknown compound: LYS
Unknown compound: MET
Unknown compound: MG
Unknown compound: PHE
Unknown compound: SER
Unknown compound: THR
Unknown compound: TRP
Unknown compound: TYR
Unknown compound: VAL

  warnings.warn(err)


In [18]:
print(len(df), "chains written")

8586 chains written
